#SOURCE MODIFIED FROM: Team, Scientific Data Curation (2019). Metadata record for: Interaction data from the Copenhagen Networks Study. figshare. Dataset. https://doi.org/10.6084/m9.figshare.11283407

# Copenhagen Networks Study: Colab-ready notebook

This copy of the course notebook runs in Google Colab. What changed from the original:

1. **Data location.** The original changed into a hard-coded instructor folder (`/content/drive/MyDrive/MSSP607/Examples`). This version mounts your Drive and reads the files from the folder you set in `DATA_DIR`. If the files are not there, it searches your MyDrive for them.
2. **File check.** Before loading anything, the notebook confirms that all five CSV files are present and explains what to fix if one is missing.
3. **Bluetooth plot.** `plt.subplots` now uses `squeeze=False`, so the function no longer crashes when a time bin has only one connected component (or none).
4. **SMS-by-gender plot.** The original drew the count matrix with senders on the rows but labelled the rows "recipient", so the shading of the two cross-gender cells did not match their printed counts. The matrix is now transposed so the shading, labels and numbers agree.
5. **Part 4 (added).** Cells that check the article's headline numbers against the released files and draw the figures used in the case study report.

## 1. Setup: mount Google Drive and point to the data

Step 1 of the assignment expands `Copenhagen Study.zip` into a folder in MyDrive. Set `DATA_DIR` to that folder.

In [ ]:
import glob
import os

try:
    import google.colab  # noqa: F401  (only importable inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Edit this line if you expanded the files into a folder with another name.
    DATA_DIR = "/content/drive/MyDrive/Copenhagen Study"
    if not os.path.exists(os.path.join(DATA_DIR, "bt_symmetric.csv")):
        hits = glob.glob("/content/drive/MyDrive/**/bt_symmetric.csv", recursive=True)
        if hits:
            DATA_DIR = os.path.dirname(hits[0])
else:
    DATA_DIR = "./"  # running outside Colab: keep this notebook next to the CSV files

print("Reading data from:", os.path.abspath(DATA_DIR))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import itertools
import math
%matplotlib inline
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 10)

In [ ]:
DATA_FILENAMES = {"bluetooth": "bt_symmetric.csv",
                  "calls": "calls.csv",
                  "sms": "sms.csv",
                  "facebook_friends": "fb_friends.csv",
                  "genders": "genders.csv"}

missing = [f for f in DATA_FILENAMES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"Missing from {os.path.abspath(DATA_DIR)}: {missing}. "
        "Expand Copenhagen Study.zip into MyDrive and set DATA_DIR in the first code cell.")

In [ ]:
def LoadData(data_filename, data_dir):
  """Loads data from |data_dir|/|data_filename|. Returns Pandas dataframe."""
  return pd.read_csv(os.path.join(data_dir, data_filename))

In [ ]:
df_bt = LoadData(DATA_FILENAMES["bluetooth"], DATA_DIR)
print("bluetooth\n", df_bt.head())

df_calls = LoadData(DATA_FILENAMES["calls"], DATA_DIR)
print("\ncalls\n", df_calls.head())

df_sms = LoadData(DATA_FILENAMES["sms"], DATA_DIR)
print("\nsms\n", df_sms.head())

df_facebook_friends = LoadData(DATA_FILENAMES["facebook_friends"], DATA_DIR)
print("\nfacebook friends\n", df_facebook_friends.head())

df_genders = LoadData(DATA_FILENAMES["genders"], DATA_DIR)
print("\ngenders\n", df_genders.head())

## 2. Bluetooth proximity network at single 5-minute bins

How the code builds the network:

* **Nodes and edges.** It keeps scans where `user_b >= 0`, i.e. both devices belong to study participants (`-1` = empty scan, `-2` = a device outside the study). Each remaining row becomes an undirected edge between two participants.
* **Edge weight.** The received signal strength (RSSI, in dBm) is the edge weight. A higher (less negative) RSSI means the phones were closer, and it is drawn as a thicker line.
* **Components.** It splits the graph into connected components and draws each one in its own panel with a spring layout.

At one 5-minute bin the network is many small, separate groups.

In [ ]:
def VisualizeBluetoothNetwork(df_bt, timestamp):
  """ Plots each connected component in |df_bt| network at |timestamp|."""
  # NetworkX graph to plot the results.
  G = nx.Graph()

  # In-study users (id >= 0) at given timestamp.
  in_study_users_at_timestamp = df_bt[(df_bt["# timestamp"] == timestamp)
                                      & (df_bt["user_b"] >= 0)][[
                                          "user_a", "user_b", "rssi"
                                      ]]

  # Builds an undirected network of the in-study users.
  for row in in_study_users_at_timestamp.itertuples():
    user_a = row[1]
    user_b = row[2]
    rssi = row[3]
    G.add_edge(user_a, user_b, weight=rssi)

  # For clarity, we consider each connected component separately.
  connected_components = [G.subgraph(c).copy() for c in nx.connected_components(G)]
  if not connected_components:
    print("No in-study proximity edges at timestamp={}".format(timestamp))
    return

  # squeeze=False keeps |axs| two-dimensional, so a single component also works.
  fig, axs = plt.subplots(1, len(connected_components),
                          figsize=(1.5 * len(connected_components), 3), squeeze=False)

  # Plots the connected components of the network.
  fig.suptitle("network components at timestamp={}".format(timestamp))
  for i, connected_component in enumerate(connected_components):
    nx.draw(
        connected_component,
        # Positions of nodes are calculated using spring layout.
        pos=nx.spring_layout(connected_component, scale=0.2, seed=1),
        ax=axs[0][i],
        # Width of the edges is based on higher RSSI = stronger edge.
        width=[(100 + connected_component[u][v]["weight"]) / 10.0
               for u, v in connected_component.edges()],
        node_size=500,
        node_color="black",
        with_labels=True,
        font_size=10,
        font_color="white")
  plt.show()


VisualizeBluetoothNetwork(df_bt, 0)
VisualizeBluetoothNetwork(df_bt, 300)
VisualizeBluetoothNetwork(df_bt, 600)

## 3. SMS communication by gender

This cell shows how the layers join on the shared user id. It merges `sms.csv` with `genders.csv` twice, once for the sender and once for the recipient. It then counts messages (`totals`) and distinct sender-to-recipient pairs (`unique_pairs`) for each gender combination.

Note: `pd.merge` is an inner join, so messages involving a participant with no gender record are dropped (23,069 of 24,333 messages remain).

In [ ]:
def VisualizeTextMessages(df_sms, df_genders):
  """Plots the SMS communication by gender of senders and recipients."""

  # Creates SMS network including gender of the senders.
  df_sms_with_gender_sender = pd.merge(
      df_sms, df_genders, left_on="sender", right_on="# user").rename(
          index=str, columns={
              "female": "female_sender"
          }).drop(
              "# user", axis=1)

  # Creates SMS network including gender of the recipients.
  df_sms_with_gender = pd.merge(
      df_sms_with_gender_sender,
      df_genders,
      left_on="recipient",
      right_on="# user").rename(
          index=str, columns={
              "female": "female_recipient"
          }).drop(
              "# user", axis=1)

  # Computes total communication (all messages) and number of unique pairs based
  # on sender and recipient gender. Rows = sender gender, columns = recipient gender.
  totals = np.zeros((2, 2))
  unique_pairs = np.zeros((2, 2))
  for female_sender in [0, 1]:
    for female_recipient in [0, 1]:
      totals[female_sender, female_recipient] = df_sms_with_gender[
          (df_sms_with_gender["female_sender"] == female_sender)
          &
          (df_sms_with_gender["female_recipient"] == female_recipient)].shape[0]
      unique_pairs[female_sender, female_recipient] = df_sms_with_gender[
          (df_sms_with_gender["female_sender"] == female_sender)
          & (df_sms_with_gender["female_recipient"] == female_recipient)][[
              "sender", "recipient"
          ]].drop_duplicates().shape[0]

  # Plots the breakdowns. matshow puts matrix rows on the y-axis, so the matrix is
  # transposed (.T) to put senders on the x-axis and recipients on the y-axis,
  # matching the axis labels and the text positions below.
  for matrix, title in [(totals, "Total SMS communication"), (unique_pairs, "Unique SMS pairs")]:
    plt.matshow(matrix.T, cmap=plt.cm.gray)
    plt.grid(False)
    plt.xlabel("sender")
    plt.ylabel("recipient")
    plt.xticks([0, 1], ["male", "female"])
    plt.yticks([0, 1], ["male", "female"])
    for sender, recipient in itertools.product([0, 1], [0, 1]):
      plt.text(sender, recipient, "n={}".format(int(matrix[sender][recipient])),
               color="red", ha="center", va="center")
    plt.title(title)
    plt.show()

  return df_sms_with_gender


df_sms_with_gender = VisualizeTextMessages(df_sms, df_genders)

## 4. Added for the case study report: checking the article's numbers

These cells reuse the data frames loaded above and produce Tables A1 to A3 and Figures A1 and A2 of the case study report: the summary counts in Tables 1 to 5 of Sapiezynski et al. (2019), the gender breakdown of text messages, and checks of several headline findings.

**Assumption.** The files only carry relative timestamps. The article says the data start on a Sunday during term, and the activity pattern fits timestamp 0 being midnight at the start of that Sunday. Days are numbered 0 = Sunday to 6 = Saturday on that basis.

In [ ]:
# Participant-to-participant proximity only: drop empty scans (-1) and outside devices (-2).
# Each pair is stored as (smaller id, larger id) so the network is undirected.
prox = df_bt[df_bt["user_b"] >= 0].rename(columns={"# timestamp": "t"})
prox = pd.DataFrame({"t": prox["t"],
                     "a": np.minimum(prox["user_a"], prox["user_b"]),
                     "b": np.maximum(prox["user_a"], prox["user_b"])})

def undirected_pairs(df, col1, col2):
  """Returns one (smaller id, larger id) row per record of a directed edge list."""
  return pd.DataFrame({"a": np.minimum(df[col1], df[col2]), "b": np.maximum(df[col1], df[col2])})

call_pairs = undirected_pairs(df_calls, "caller", "callee")
sms_pairs = undirected_pairs(df_sms, "sender", "recipient")
N_BINS = 4 * 7 * 24 * 12          # 8,064 five-minute bins in four weeks
N_PARTICIPANTS = 706              # participants in the Bluetooth data
POSSIBLE_PAIRS = N_PARTICIPANTS * (N_PARTICIPANTS - 1) / 2

In [ ]:
# Table A1: records and users per layer, as published (Tables 1-5) and as found in the files.
users = {
    "Bluetooth proximity": set(df_bt["user_a"]) | set(df_bt.loc[df_bt["user_b"] >= 0, "user_b"]),
    "Phone calls": set(df_calls["caller"]) | set(df_calls["callee"]),
    "Text messages (SMS)": set(df_sms["sender"]) | set(df_sms["recipient"]),
    "Facebook friendships": set(df_facebook_friends["# user_a"]) | set(df_facebook_friends["user_b"]),
    "Gender": set(df_genders["# user"]),
}
records = {"Bluetooth proximity": len(df_bt), "Phone calls": len(df_calls),
           "Text messages (SMS)": len(df_sms), "Facebook friendships": len(df_facebook_friends),
           "Gender": len(df_genders)}
published = {"Bluetooth proximity": (5474289, 706), "Phone calls": (3600, 540),
             "Text messages (SMS)": (24333, 577), "Facebook friendships": (6429, 800),
             "Gender": (788, 788)}
table_a1 = pd.DataFrame({
    "records (article)": {k: v[0] for k, v in published.items()},
    "records (files)": records,
    "users (article)": {k: v[1] for k, v in published.items()},
    "users (files)": {k: len(v) for k, v in users.items()},
})
print(table_a1)
print("\nBluetooth rows: participant pairs {:,}, empty scans {:,}, outside devices {:,}".format(
    (df_bt["user_b"] >= 0).sum(), (df_bt["user_b"] == -1).sum(), (df_bt["user_b"] == -2).sum()))
print("Women among participants with a gender record: {:.1%}".format(df_genders["female"].mean()))
print("Missed calls: {:,} of {:,}".format((df_calls["duration"] == -1).sum(), len(df_calls)))

In [ ]:
# Table A3 (availability and density). Data availability: share of the 8,064 bins in which a participant scanned or was scanned.
seen = pd.concat([
    df_bt[["# timestamp", "user_a"]].set_axis(["t", "u"], axis=1),
    df_bt.loc[df_bt["user_b"] >= 0, ["# timestamp", "user_b"]].set_axis(["t", "u"], axis=1),
]).drop_duplicates()
availability = seen.groupby("u").size() / N_BINS
print("Median data availability per participant: {:.3f}  (article: 0.81)".format(availability.median()))

# Density of the aggregated proximity network: share of all possible pairs ever in contact.
pairs_4wk = prox[["a", "b"]].drop_duplicates()
print("Four-week density: {:.1%}  (article: 30%)".format(len(pairs_4wk) / POSSIBLE_PAIRS))
week = prox["t"] // (7 * 86400)
for w in range(4):
  d = len(prox.loc[week == w, ["a", "b"]].drop_duplicates()) / POSSIBLE_PAIRS
  print("  week {}: {:.1%}".format(w + 1, d))
print("  (article: weekly aggregates 10% to 20%)")

In [ ]:
# Table A3 (cross-channel overlap): do people who text or call each other also meet in person?
bins_together = prox.groupby(["a", "b"]).size().sort_values(ascending=False)
met = set(bins_together.index)

def share_in(pairs, pair_set):
  return np.mean([p in pair_set for p in zip(pairs["a"], pairs["b"])])

calling = call_pairs.drop_duplicates()
print("Calling pairs ever in Bluetooth proximity: {:,} of {:,} = {:.1%}  (article: 89% of calling pairs)".format(
    sum(p in met for p in zip(calling["a"], calling["b"])), len(calling), share_in(calling, met)))
print("Calls made by pairs ever in Bluetooth proximity: {:.1%}  (matches the article's 89% when weighted by calls)".format(
    share_in(call_pairs, met)))
print("SMS between pairs ever in Bluetooth proximity:   {:.1%}".format(share_in(sms_pairs, met)))
for top in [0.01, 0.15]:
  k = int(round(top * len(bins_together)))
  top_pairs = set(bins_together.index[:k])
  print("Top {:.0%} of proximity pairs ({:,} pairs): {:.1%} of SMS, {:.1%} of calls".format(
      top, k, share_in(sms_pairs, top_pairs), share_in(call_pairs, top_pairs)))
print("(article: top 1% -> 56% of SMS and 47% of calls; top 15% -> over 80% of SMS)")
print("Texts per call: {:.1f}".format(len(df_sms) / len(df_calls)))
print("Distinct pairs: called {:,}, texted {:,}, ever in Bluetooth contact {:,}".format(
    len(call_pairs.drop_duplicates()), len(sms_pairs.drop_duplicates()), len(met)))

In [ ]:
# Table A3 (temporal aggregation): 40 weekday start times (10:00 and 13:00) aggregated over longer windows,
# plus one 24-hour window per weekday (20 in all).
def window_graph(t0, minutes):
  e = prox[(prox["t"] >= t0) & (prox["t"] < t0 + minutes * 60)][["a", "b"]].drop_duplicates()
  G = nx.Graph()
  G.add_edges_from(zip(e["a"], e["b"]))
  return G

rows = []
for w in range(4):
  for day in range(1, 6):                      # Monday to Friday
    for hour in [10, 13]:
      t0 = (w * 7 + day) * 86400 + hour * 3600
      windows = [(t0, m) for m in [5, 10, 20, 40, 60]]
      if hour == 10:                           # one 24-hour window per weekday, from midnight
        windows.append(((w * 7 + day) * 86400, 24 * 60))
      for start, minutes in windows:
        G = window_graph(start, minutes)
        largest = max(len(c) for c in nx.connected_components(G))
        rows.append({"window (min)": minutes, "active participants": G.number_of_nodes(),
                     "groups": nx.number_connected_components(G),
                     "largest group share": largest / G.number_of_nodes(),
                     "mean contacts per participant": 2 * G.number_of_edges() / N_PARTICIPANTS})
aggregation = pd.DataFrame(rows).groupby("window (min)").median()
print(aggregation.round(2))
print("(article: average degree grows from about 2 to about 30 as windows grow to 24 hours)")

In [ ]:
# Figure A1: one typical weekday window drawn at 5 and at 40 minutes, with the same node positions.
from matplotlib.lines import Line2D
BLUE, GREY, FAINT, EDGE, SECONDARY = "#2a78d6", "#8f8d87", "#dcdad3", "#cfcdc6", "#52514e"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9})

t0 = (2 * 7 + 1) * 86400 + 13 * 3600           # week 3, Monday 13:00 (a typical weekday window)
G5, G40 = window_graph(t0, 5), window_graph(t0, 40)

# Shared node positions: the 40-minute window's largest group on the left,
# its smaller groups in a grid on the right. The 5-minute graph reuses them.
comps = sorted(nx.connected_components(G40), key=len, reverse=True)
pos = nx.kamada_kawai_layout(G40.subgraph(comps[0]))
xs, ys = zip(*pos.values())
pos = {n: ((x - min(xs)) / (max(xs) - min(xs)), (y - min(ys)) / (max(ys) - min(ys))) for n, (x, y) in pos.items()}
# Second-largest group top right; the remaining small groups in a grid beneath it.
second = nx.kamada_kawai_layout(G40.subgraph(comps[1]), center=(1.33, 0.86), scale=0.15)
pos.update(second)
ncol, rest = 4, comps[2:]
nrow = math.ceil(len(rest) / ncol)
for i, comp in enumerate(rest):
    cx, cy = 1.14 + (i % ncol) * 0.115, 0.62 - (i // ncol) * (0.62 / max(nrow - 1, 1))
    pos.update(nx.spring_layout(G40.subgraph(comp), seed=1, center=(cx, cy), scale=0.035))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.35))
for ax, G, label in [(axes[0], G5, "(a) 5-minute window"), (axes[1], G40, "(b) 40-minute window")]:
    largest = max(nx.connected_components(G), key=len)
    if G is G5:   # people who appear only later in the 40-minute window
        later = [n for n in G40.nodes() if n not in G5]
        nx.draw_networkx_nodes(G40, pos, nodelist=later, ax=ax, node_color=FAINT, node_size=7, linewidths=0)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=EDGE, width=0.35)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=[BLUE if n in largest else GREY for n in G.nodes()],
                           node_size=9, linewidths=0)
    ax.set_title("{}\n{} participants in contact, {} groups,\nlargest group {} ({:.0%})".format(
        label, G.number_of_nodes(), nx.number_connected_components(G), len(largest),
        len(largest) / G.number_of_nodes()), fontsize=9, loc="left", color="#0b0b0b")
    ax.set_xlim(-0.04, 1.62); ax.set_ylim(-0.04, 1.04); ax.set_aspect("equal"); ax.axis("off")

key = [Line2D([], [], marker="o", ls="", color=BLUE, markersize=5, label="Largest connected group"),
       Line2D([], [], marker="o", ls="", color=GREY, markersize=5, label="Other groups"),
       Line2D([], [], marker="o", ls="", color=FAINT, markersize=5, label="Not yet in contact (panel a)")]
fig.legend(handles=key, loc="lower left", ncol=3, frameon=False, fontsize=8, labelcolor=SECONDARY,
           bbox_to_anchor=(0.0, 0.0), handletextpad=0.3, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.05, 1, 1), w_pad=1.5)
plt.show()

In [ ]:
# Figure A2: share of each channel's weekly activity by hour of the week (four-week average).
INK, SECONDARY, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
SERIES = {"Physical proximity (Bluetooth pairs)": ("#2a78d6", prox["t"]),
          "Text messages": ("#eb6834", df_sms["timestamp"]),
          "Phone calls": ("#1baf7a", df_calls["timestamp"])}

fig, ax = plt.subplots(figsize=(7.2, 3.2))
for name, (color, t) in SERIES.items():
  hour_of_week = (t // 3600) % 168
  share = hour_of_week.value_counts().reindex(range(168), fill_value=0) / len(t) * 100
  smooth = pd.concat([share.iloc[-1:], share, share.iloc[:1]]).rolling(3, center=True).mean().iloc[1:-1]
  ax.plot(range(168), smooth.values, color=color, linewidth=2, solid_capstyle="round", label=name)
for d in range(1, 7):
  ax.axvline(d * 24, color=GRID, linewidth=0.8, zorder=0)
ax.set_xticks([d * 24 + 12 for d in range(7)])
ax.set_xticklabels(["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"], color=SECONDARY)
ax.tick_params(length=0, colors=SECONDARY)
ax.set_xlim(0, 168)
ax.set_ylim(0, None)
ax.set_ylabel("Share of weekly activity (%)", color=SECONDARY)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
for side in ["top", "right", "left"]:
  ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#c3c2b7")
ax.legend(loc="upper left", frameon=False, ncol=3, fontsize=8, bbox_to_anchor=(0, 1.13))
# Direct labels where each line is easiest to tell apart (text stays in ink colour).
for x, y, text in [(38.5, 2.38, "proximity peaks in\nweekday working hours"),
                   (2, 1.66, "texts and calls\ncontinue into\nevenings and\nweekends"),
                   (137.5, 2.33, "calls")]:
  ax.text(x, y, text, fontsize=7.5, color=SECONDARY, va="top")
fig.tight_layout()
plt.show()

# Table A3 (timing): share of each channel's activity in weekday evenings (18:00-24:00) or at weekends.
for name, (color, t) in SERIES.items():
  day, hour = (t // 86400) % 7, (t // 3600) % 24
  off_hours = ((day == 0) | (day == 6) | (hour >= 18)).mean()
  print("{}: {:.1%} in weekday evenings or at weekends".format(name, off_hours))

In [ ]:
# Table A2: who texts whom, by gender (from the merged frame returned in section 3).
labels = {0: "men", 1: "women"}
counts = df_sms_with_gender.groupby(["female_sender", "female_recipient"]).size()
pairs = df_sms_with_gender.drop_duplicates(["sender", "recipient"]).groupby(
    ["female_sender", "female_recipient"]).size()
table_a3 = pd.DataFrame({
    "messages": counts, "share of messages": (counts / counts.sum()).round(3),
    "distinct pairs": pairs}).rename(index=labels)
print(table_a3)
sent_by_women = counts.loc[1].sum() / counts.sum()
print("\nWomen are {:.0%} of participants with a gender record but sent {:.0%} of these messages.".format(
    df_genders["female"].mean(), sent_by_women))